# Module 11.2 — Molecular Dynamics Simulation Basics for ML Engineers
## From Physics to Machine Learning: Understanding the Data Your Models Are Trained On

## TL;DR — Plain English
**What this notebook does:** Teaches you how MD simulations work *from first principles*, so you understand where training data for models like ESMFold, AlphaFold3, and OpenFold comes from — and why membrane protein simulations are uniquely hard.

**Why this matters for computational biology ML teams interviews:** You will be asked: "Where does your training data come from?" and "Why is MD data for GPCRs different from soluble protein data?" This notebook gives you those answers with working code.

**What you'll be able to do after this:**
- Explain the Verlet integrator and force field in one sentence each
- Simulate a toy protein in Python (no GROMACS needed)
- Explain why membrane proteins need PME electrostatics and periodic boundary conditions
- Describe how ML potentials (ANI, NequIP, MACE) replace classical force fields
- Answer: "How do you validate an MD trajectory for ML training?"


## 🧬 Molecular Dynamics Simulation — Concepts for Beginners

| Term | Plain English |
|------|--------------|
| **Force field** | A set of equations and parameters that approximate the potential energy of atoms (bonds, angles, van der Waals, electrostatics) |
| **Integrator** | Algorithm that advances atom positions over time by solving Newton's equations (e.g., Verlet, leapfrog) |
| **Timestep** | The time increment per simulation step; typically 1-2 femtoseconds (too large → simulation explodes) |
| **Ensemble** | The statistical-mechanical conditions: NVT (constant temperature), NPT (constant temperature and pressure) |
| **PBC** | Periodic Boundary Conditions — the simulation box repeats infinitely in all directions to avoid edge effects |
| **PME** | Particle Mesh Ewald — an efficient algorithm for computing long-range electrostatic interactions in periodic systems |
| **Thermostat** | Algorithm that maintains constant temperature by rescaling velocities (Berendsen, Nose-Hoover, Langevin) |
| **Barostat** | Algorithm that maintains constant pressure by adjusting the simulation box volume |
| **Equilibration** | Initial simulation phase where the system relaxes to the target temperature/pressure before production data collection |
| **Trajectory** | The recorded sequence of atom positions (and velocities) over time — the raw output of an MD simulation |
| **RMSF** | Root Mean Square Fluctuation — per-residue flexibility measure; high RMSF = flexible loop, low = rigid core |
| **Coarse-grained** | A simplified representation where groups of atoms are merged into single beads — enables longer timescales |

## Prerequisites & Learning Path
| | |
|--|--|
| Prerequisites | 11/01 (Membrane Protein Dynamics), 03/01 (PDB, RMSD), 06/01 (equivariance) |
| Feeds into | 12/01 (Diffusion — trajectories as training data), 10/01 (Fine-tuning with MD features) |
| Difficulty | ⭐⭐⭐ (moderate — physics + code) |

### Free Resources for This Notebook
- **OpenMM documentation** (openmm.org/documentation) — the Python MD engine used in practice
- **GROMACS tutorial** — Justin Lemkul's free tutorial at `mdtutorials.com` is the standard
- **MIT 8.592J** Statistical Physics in Biology (OCW) — covers the statistical mechanics behind MD
- **Frenkel & Smit** "Understanding Molecular Simulation" — free PDF via university access; Chapter 3 (Verlet), Chapter 13 (membrane)
- **nglview** — in-browser MD trajectory viewer for Jupyter: `pip install nglview`
- **MDAnalysis** — Python library for trajectory analysis: `mdanalysis.org`


## 🟢 Complete Beginner's Guide

**Assumed background:** Zero physics background required.

### What you need to know before starting

- **Atom** — the smallest unit of matter. Proteins are made of atoms (carbon, nitrogen, oxygen, hydrogen, sulfur).
- **Force** — a push or pull on an object. In proteins, atoms attract or repel each other based on their charges, distances, and bond types.
- **Newton's 2nd Law in protein context** — F = ma. If we know the force on every atom, we can calculate how fast it accelerates (a = F/m) and then update its position over a tiny time step (femtoseconds = 10^-15 s).
- **Molecular Dynamics (MD) simulation** — repeated application of Newton's 2nd law: compute forces → update velocities → update positions → repeat millions of times.
- **Force field** — a set of equations and parameters that define how atoms interact (bonds, angles, torsions, electrostatics, van der Waals).
- **Trajectory** — the time series of all atom positions saved during a simulation; this is the main output you analyze.

### Vocabulary you MUST know

| Term | One-line definition |
|------|--------------------|
| `RMSD` | Root Mean Square Deviation — measures how much structure moved from reference |
| `RMSF` | Per-residue flexibility — which parts of the protein move most |
| `PBC (Periodic Boundary Conditions)` | Simulation trick: wrap the box to avoid edge effects |
| `equilibration` | Warming up phase before production simulation |
| `NPT / NVT ensemble` | Statistical ensembles controlling temperature and pressure |
| `lipid bilayer` | Two-layer membrane of phospholipids that membrane proteins sit inside |

### 3-Step Reading Strategy for Beginners

1. **Read all markdown cells first** — focus on what biological question the simulation answers.
2. **Run code cells one at a time** — visualize the protein and trajectory at each step.
3. **Modify one number and re-run** — change the RMSD calculation reference frame and see how the plot changes.

### If you're stuck

- **YouTube:** 'Introduction to Molecular Dynamics' by GROMACS team (https://www.youtube.com/watch?v=8hD_BjY7GQo)
- **YouTube:** 'Protein MD simulation tutorial' by Justin Lemkul (GROMACS tutorial series)
- **Book:** *Understanding Molecular Simulation* by Frenkel & Smit, Chapters 1–3.
- **Web:** http://www.mdtutorials.com/ — free GROMACS tutorials from scratch.


## Section 1 — What Is Molecular Dynamics?

### The Core Idea

MD simulation integrates Newton's second law (F = ma) for every atom, at every timestep (typically 2 fs):

```
x(t + Δt) = 2x(t) - x(t - Δt) + (F(t)/m)·Δt²    ← Verlet integrator
```

The force F comes from a **force field** — an empirical potential energy function:

```
V_total = V_bonds + V_angles + V_dihedrals + V_LJ + V_electrostatic
```

**Why ML engineers need to understand this:**
1. Your training data (PDB structures) are snapshots from MD or X-ray crystallography
2. MD-derived structural ensembles tell you which conformations are thermodynamically accessible
3. ML potentials (Section 6) replace the classical force field with a neural network — you need to know what you're replacing


## 📖 Prerequisites & Cross-References

**Before this notebook, you should be comfortable with:**
- **Membrane proteins** — TM topology, hydrophobicity, lipid bilayer context. *Review: `11_membrane_protein_dynamics/01_membrane_protein_dynamics.ipynb`*
- **3D coordinates** — PDB parsing, distance computation. *Review: `03_protein_structural_biology/01_protein_structure.ipynb`*

**Quick recap of terms used here:**
- **RMSD** — average distance between equivalent atoms in two structures.
- **Kabsch** — algorithm finding optimal rotation to minimize RMSD.
- **embedding** — converting atoms/residues to numerical vectors. *See `03/01`.*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import List, Tuple

# ─────────────────────────────────────────────────────────────────────
# TOY MD SIMULATION: A 1D Lennard-Jones chain
# This is a minimal MD engine — no force fields, just physics
# Real engines (OpenMM, GROMACS) do this for 100,000+ atoms in parallel
# ─────────────────────────────────────────────────────────────────────

@dataclass
class Particle:
    pos: float      # position in Angstroms
    vel: float      # velocity in Ang/ps
    mass: float     # in amu

def lj_force_1d(positions: np.ndarray, epsilon: float = 1.0, sigma: float = 3.8) -> np.ndarray:
    # Lennard-Jones: V(r) = 4ε[(σ/r)^12 - (σ/r)^6]
    # F = -dV/dr
    n = len(positions)
    forces = np.zeros(n)
    for i in range(n):
        for j in range(i+1, n):
            r = abs(positions[j] - positions[i])
            if r < 0.1:  # avoid singularity
                r = 0.1
            r6 = (sigma / r) ** 6
            f_mag = 24 * epsilon / r * (2 * r6**2 - r6)
            direction = np.sign(positions[j] - positions[i])
            forces[i] -= f_mag * direction  # force on i from j
            forces[j] += f_mag * direction  # Newton's 3rd law
    return forces

def verlet_step(positions: np.ndarray, velocities: np.ndarray, masses: np.ndarray,
                dt: float = 0.002) -> Tuple[np.ndarray, np.ndarray]:
    # Velocity Verlet integrator — symplectic, time-reversible, energy-conserving
    forces = lj_force_1d(positions)
    accelerations = forces / masses

    new_positions = positions + velocities * dt + 0.5 * accelerations * dt**2
    new_forces = lj_force_1d(new_positions)
    new_accelerations = new_forces / masses
    new_velocities = velocities + 0.5 * (accelerations + new_accelerations) * dt

    return new_positions, new_velocities

# Initialize: 5-residue peptide backbone Cα atoms
n_atoms = 5
np.random.seed(42)
positions = np.array([3.8 * i for i in range(n_atoms)], dtype=float)  # Å
velocities = np.random.normal(0, 0.1, n_atoms)  # thermal motion
masses = np.ones(n_atoms) * 110.0  # average residue mass in amu

# Run 500 steps (1 ps)
n_steps = 500
dt = 0.002  # 2 fs
traj = np.zeros((n_steps, n_atoms))
kinetic = np.zeros(n_steps)

for step in range(n_steps):
    traj[step] = positions
    KE = 0.5 * np.sum(masses * velocities**2)
    kinetic[step] = KE
    positions, velocities = verlet_step(positions, velocities, masses, dt)

print("TOY 1D MD SIMULATION")
print(f"  Atoms: {n_atoms} Cα backbone atoms")
print(f"  Steps: {n_steps} × {dt*1000:.0f} fs = {n_steps*dt:.1f} ps total")
print(f"  Initial positions: {traj[0].round(2)}")
print(f"  Final positions:   {traj[-1].round(2)}")
print(f"  Kinetic energy fluctuation: {kinetic.std():.2f} (lower = better conserved)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Trajectory
for i in range(n_atoms):
    axes[0].plot(traj[:, i], label=f'Atom {i}', alpha=0.7)
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Position (Å)')
axes[0].set_title('Cα Positions Over Time (1D MD)')
axes[0].legend()

# Energy
axes[1].plot(kinetic, color='red', alpha=0.7)
axes[1].set_xlabel('Step')
axes[1].set_ylabel('Kinetic Energy (amu·Å²/ps²)')
axes[1].set_title('Kinetic Energy (should fluctuate, not drift)')

plt.tight_layout()
plt.savefig('/tmp/md_toy_traj.png', dpi=80, bbox_inches='tight')
plt.show()
print("\nLesson: Verlet integrator conserves energy — KE fluctuates but doesn't drift upward")


> ⚠️ **Common Mistake:** Setting the integration timestep too large, causing energy explosion
>
> **Wrong:** `dt = 0.01  # 10 fs -- atoms overlap, forces spike, simulation explodes`
> **Right:** `dt = 0.002  # 2 fs -- standard for all-atom MD with constrained H-bonds`
> **Why:** The Verlet integrator assumes forces are approximately constant within one timestep. Large dt violates this for fast bond vibrations (especially O-H, N-H), causing numerical instability.

> **Expected output:** Toy 1D MD simulation output: atom count, timestep info, initial/final positions, and kinetic energy fluctuation (lower = better energy conservation)  
> If you see this, your code is working correctly.  
> If you see an error, check the Troubleshooting section or re-run the cell.

## Section 2 — Force Fields: What They Are and Why They're Approximate

### The Force Field Equation

```
V = Σ_bonds k_b(r - r_0)²           ← harmonic bond stretching
  + Σ_angles k_θ(θ - θ_0)²          ← angle bending
  + Σ_dihedrals V_n/2 [1+cos(nφ-γ)] ← torsion (this is why your protein folds!)
  + Σ_i<j 4ε[(σ/r)^12 - (σ/r)^6]   ← Lennard-Jones (van der Waals)
  + Σ_i<j q_iq_j / (4πε_0 r)        ← Coulomb electrostatics
```

**Common force fields:**
| Force Field | Best For | Key Feature |
|-------------|----------|-------------|
| AMBER ff19SB | Soluble proteins | Best-validated for protein folding |
| CHARMM36m | Membrane proteins | Lipid bilayer parameters included |
| OPLS-AA | Small molecules / drugs | Good for ligand binding |
| MARTINI | Coarse-grained MD | 4 heavy atoms → 1 bead; 1000× faster |

**Why force fields matter for ML:**
- AF3 was trained on structures relaxed with AMBER
- Force field choice affects which conformations you sample
- Membrane proteins need **CHARMM36m** — soluble protein force fields don't have lipid parameters


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ─────────────────────────────────────────────────────────────────────
# FORCE FIELD COMPONENTS — visualizing each term
# ─────────────────────────────────────────────────────────────────────

r = np.linspace(0.5, 10, 500)  # distance in Angstroms
theta = np.linspace(0, np.pi, 500)  # angle in radians
phi = np.linspace(-np.pi, np.pi, 500)  # dihedral in radians

# LJ potential
def lennard_jones(r, epsilon=0.1, sigma=3.8):
    return 4 * epsilon * ((sigma/r)**12 - (sigma/r)**6)

# Harmonic bond
def harmonic_bond(r, k_b=500.0, r0=3.8):
    return k_b * (r - r0)**2

# Torsion (Ramachandran-relevant backbone dihedral)
def torsion(phi, V_n=1.0, n=3, gamma=0.0):
    return V_n/2 * (1 + np.cos(n * phi - gamma))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# LJ
lj = lennard_jones(r)
lj_clipped = np.clip(lj, -0.2, 0.5)
axes[0].plot(r, lj_clipped, 'b-', linewidth=2)
axes[0].axhline(0, color='gray', linestyle='--', alpha=0.5)
axes[0].axvline(3.8, color='red', linestyle=':', label='σ (3.8 Å, Cα-Cα)')
axes[0].set_xlabel('Distance (Å)')
axes[0].set_ylabel('V (kcal/mol)')
axes[0].set_title('Lennard-Jones\n(van der Waals)')
axes[0].legend()
axes[0].set_ylim(-0.2, 0.5)

# Harmonic bond (zoomed near equilibrium)
r_bond = np.linspace(3.0, 4.6, 200)
axes[1].plot(r_bond, harmonic_bond(r_bond, k_b=2.0, r0=3.8), 'g-', linewidth=2)
axes[1].axvline(3.8, color='red', linestyle=':', label='r₀ = 3.8 Å (Cα-Cα)')
axes[1].set_xlabel('Bond Length (Å)')
axes[1].set_ylabel('V (kcal/mol)')
axes[1].set_title('Harmonic Bond\n(stiff spring)')
axes[1].legend()

# Torsion (backbone phi/psi)
axes[2].plot(np.degrees(phi), torsion(phi, V_n=1.0, n=1), label='n=1 (broad)', alpha=0.8)
axes[2].plot(np.degrees(phi), torsion(phi, V_n=0.5, n=3), label='n=3 (three barriers)', alpha=0.8)
axes[2].set_xlabel('Dihedral φ (degrees)')
axes[2].set_ylabel('V (kcal/mol)')
axes[2].set_title('Torsion Potential\n(controls α-helix vs β-sheet)')
axes[2].legend()

plt.tight_layout()
plt.savefig('/tmp/md_force_field.png', dpi=80, bbox_inches='tight')
plt.show()

print("FORCE FIELD INSIGHT:")
print("The torsion potential with n=3 creates THREE preferred dihedral angles.")
print("The Ramachandran plot (phi/psi angles) is essentially a 2D torsion energy landscape.")
print("AF3 learns this landscape implicitly from PDB structures — it doesn't use force fields.")
print()
print("KEY FACT: CHARMM36m has explicit lipid parameters (DPPC, POPE, POPC, cholesterol).")
print("AMBER ff19SB has NO lipid parameters — it will give garbage for membrane proteins.")


## Section 3 — Periodic Boundary Conditions and Long-Range Electrostatics

### Why PBC Matters for Membrane Proteins

Without periodic boundary conditions (PBC), you're simulating a protein in a vacuum. The membrane bilayer **must be simulated with PBC** because:
1. The membrane extends infinitely in the xy-plane
2. Without PBC, edge effects dominate
3. Water molecules evaporate from the surface

### The PME Problem

Long-range electrostatics (Coulomb) decay as 1/r — you can't just cut them off at 12 Å. For charged residues and phospholipid head groups, the electrostatic contribution extends over 50+ Å.

**Particle Mesh Ewald (PME):** splits electrostatics into short-range (direct space) + long-range (reciprocal space via FFT). This is why membrane MD is 10× more expensive than soluble protein MD.

```
E_total = E_direct(r < r_cut) + E_reciprocal(k-space) + E_self correction
```

For ML: if your training data includes membrane proteins from CHARMM simulations, the model has seen structures shaped by PME electrostatics. If you switch to AMBER without PME-equivalent treatment, you're in distribution shift territory.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ─────────────────────────────────────────────────────────────────────
# PERIODIC BOUNDARY CONDITIONS — the "minimum image convention"
# ─────────────────────────────────────────────────────────────────────

# --- minimum_image() ---
def minimum_image(dx: float, box_length: float) -> float:
    # Map displacement into [-L/2, L/2]
    return dx - box_length * np.round(dx / box_length)

# --- setup_membrane_box() ---
def setup_membrane_box(n_lipids_per_leaflet: int = 50,
                       box_xy: float = 60.0,  # Å
                       box_z: float = 80.0) -> dict:
    # Typical GPCR simulation box: 90×90×110 Å for one GPCR
    # Here we use a smaller box for illustration
    np.random.seed(42)

    # Lipid head groups (simplified — real DPPC has 130 atoms)
    upper_leaflet_z = box_z * 0.65  # upper membrane at 65% of box height
    lower_leaflet_z = box_z * 0.35  # lower membrane at 35%

    upper_heads = np.column_stack([
        np.random.uniform(0, box_xy, n_lipids_per_leaflet),
        np.random.uniform(0, box_xy, n_lipids_per_leaflet),
        np.full(n_lipids_per_leaflet, upper_leaflet_z) + np.random.normal(0, 1.0, n_lipids_per_leaflet)
    ])

    lower_heads = np.column_stack([
        np.random.uniform(0, box_xy, n_lipids_per_leaflet),
        np.random.uniform(0, box_xy, n_lipids_per_leaflet),
        np.full(n_lipids_per_leaflet, lower_leaflet_z) + np.random.normal(0, 1.0, n_lipids_per_leaflet)
    ])

    # Protein (GPCR transmembrane helices — 7 TM helices, simplified as cylinders)
    tm_helix_centers_xy = np.array([
        [28, 28], [32, 22], [38, 25], [40, 31], [36, 37], [30, 37], [26, 33]
    ])
    tm_helices = []
    # Loop over xy
    for xy in tm_helix_centers_xy:
        z_range = np.linspace(lower_leaflet_z + 2, upper_leaflet_z - 2, 15)
        # Loop over z
        for z in z_range:
            tm_helices.append([xy[0] + np.random.normal(0, 0.5),
                               xy[1] + np.random.normal(0, 0.5), z])
    tm_helices = np.array(tm_helices)

    # Return result
    return {
        'box': (box_xy, box_xy, box_z),
        'upper_heads': upper_heads,
        'lower_heads': lower_heads,
        'tm_helices': tm_helices,
        'upper_z': upper_leaflet_z,
        'lower_z': lower_leaflet_z
    }

box = setup_membrane_box()

# Create figure
fig = plt.figure(figsize=(14, 5))

# XZ cross-section
ax1 = fig.add_subplot(121)
ax1.scatter(box['upper_heads'][:, 0], box['upper_heads'][:, 2],
            c='blue', s=30, alpha=0.6, label='Upper leaflet heads (PO4)')
ax1.scatter(box['lower_heads'][:, 0], box['lower_heads'][:, 2],
            c='blue', s=30, alpha=0.6, label='Lower leaflet heads (PO4)')
ax1.scatter(box['tm_helices'][:, 0], box['tm_helices'][:, 2],
            c='orange', s=10, alpha=0.5, label='TM helix Cα')
ax1.axhline(box['upper_z'], color='blue', linestyle='--', alpha=0.3)
ax1.axhline(box['lower_z'], color='blue', linestyle='--', alpha=0.3)
ax1.set_xlabel('X (Å)')
ax1.set_ylabel('Z (Å)')
ax1.set_title('XZ Cross-Section: GPCR in Lipid Bilayer\n(PBC wraps at X=0 and X=60 Å)')
ax1.legend(fontsize=8)
ax1.set_xlim(-5, 65)

# Top-down XY view of bilayer
ax2 = fig.add_subplot(122)
ax2.scatter(box['upper_heads'][:, 0], box['upper_heads'][:, 1],
            c='blue', s=50, alpha=0.5, label='Lipid head groups (top view)')
# TM helix cross-sections (circles)
# Loop over xy
for xy in [[28,28],[32,22],[38,25],[40,31],[36,37],[30,37],[26,33]]:
    circle = plt.Circle(xy, 2, color='orange', alpha=0.7)
    ax2.add_patch(circle)
ax2.set_xlabel('X (Å)')
ax2.set_ylabel('Y (Å)')
ax2.set_title('XY Top View: 7-TM Helix Bundle\nin Lipid Bilayer (PBC box shown)')
ax2.set_aspect('equal')
ax2.set_xlim(0, 60)
ax2.set_ylim(0, 60)
# PBC box boundary
# Loop over x
for x in [0, 60]:
    ax2.axvline(x, color='gray', linestyle='--', alpha=0.5)
# Loop over y
for y in [0, 60]:
    ax2.axhline(y, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('/tmp/md_membrane_box.png', dpi=80, bbox_inches='tight')
# Display the plot
plt.show()

print("MEMBRANE SIMULATION BOX FACTS:")
print(f"  Box size: {box['box'][0]:.0f} × {box['box'][1]:.0f} × {box['box'][2]:.0f} Å")
print(f"  Bilayer thickness: {box['upper_z'] - box['lower_z']:.1f} Å (real DPPC: ~38 Å)")
print(f"  TM helix Cα atoms modeled: {len(box['tm_helices'])}")
print()
print("PBC minimum image convention check:")
box_length = 60.0
test_pairs = [(5, 55), (10, 50), (1, 59)]
# Loop over p1, p2
for p1, p2 in test_pairs:
    naive_dist = abs(p2 - p1)
    pbc_dist = abs(minimum_image(p2 - p1, box_length))
    print(f"  Atoms at x={p1} and x={p2}: naive={naive_dist:.0f} Å, PBC={pbc_dist:.0f} Å")
print()
print("Key insight: atoms at x=5 and x=55 are CLOSER than naive distance suggests under PBC")


## 🏁 Checkpoint -- Pause and Verify

Before continuing, make sure you can:
1. Explain the Verlet integration algorithm and why it conserves energy better than Euler
2. Describe periodic boundary conditions (PBC) and why they are needed for MD simulations
3. List the key force field terms (bonds, angles, dihedrals, Lennard-Jones, Coulomb) and what each models

**If any of these feel shaky**, re-read the section above or review the prerequisite notebook listed in the Cross-References section.

## Section 4 — MD Trajectories as ML Training Data

### The Data Pipeline: MD → ML

```
Wet lab experiment
    ↓ (X-ray / cryo-EM)
PDB structure (single snapshot)
    ↓ (add hydrogens, lipids, water)
MD simulation (GROMACS / OpenMM / AMBER)
    ↓ (10-100 μs of sampling)
Trajectory (.xtc / .dcd files)
    ↓ (MDAnalysis / MDTraj)
Structural ensemble (10,000-100,000 frames)
    ↓ (feature extraction)
ML training data
```

### What Makes Good MD Training Data

| Quality Check | How to Measure | Red Flag |
|---------------|----------------|----------|
| RMSD convergence | RMSD from crystal structure vs time | Continuous drift (not equilibrated) |
| Ramachandran coverage | phi/psi population | >5% in disallowed region |
| Secondary structure stability | DSSP over time | Helix fraying in first 10 ns |
| Lipid order parameter | S_CD for each carbon | Negative for membrane disordering |
| Protein-lipid contacts | Contact frequency map | Zero contacts (protein escaped bilayer) |
| System density | box volume over time | Density drift = poor barostat |


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ─────────────────────────────────────────────────────────────────────
# TRAJECTORY QUALITY CHECKS — what you run BEFORE using MD data for ML
# ─────────────────────────────────────────────────────────────────────

np.random.seed(42)
n_frames = 500  # 500 ns trajectory, 1 frame per ns

# 1. RMSD from crystal structure (Cα)
# Good: fast rise then plateau. Bad: continuous drift
t = np.linspace(0, 500, n_frames)
rmsd_equilibration = 2.0 * (1 - np.exp(-t/20))    # equilibrates at ~2 Å
rmsd_noise = np.random.normal(0, 0.15, n_frames)
rmsd = rmsd_equilibration + rmsd_noise
rmsd = np.abs(rmsd)

# 2. Secondary structure stability (fraction helical over time)
helix_frac = 0.65 - 0.1 * np.exp(-t/30) + np.random.normal(0, 0.03, n_frames)

# 3. Lipid order parameter S_CD (each acyl chain carbon)
# Perfect crystal: S_CD = 0.5; fluid bilayer: 0.1-0.3; disordered: <0.1
n_carbons = 16  # DPPC has 16 carbons per acyl chain
carbon_pos = np.arange(1, n_carbons + 1)
s_cd = 0.32 * np.exp(-carbon_pos * 0.08) + 0.05 + np.random.normal(0, 0.02, n_carbons)
s_cd_ideal = 0.30 * np.exp(-carbon_pos * 0.08) + 0.05

# 4. Protein-lipid contact frequency
n_residues = 50  # simplified TM helix
contact_freq = np.zeros(n_residues)
# Transmembrane residues (15-40) have high contact frequency
contact_freq[14:41] = 0.3 + 0.3 * np.random.random(27)
contact_freq[:14] = 0.05 * np.random.random(14)
contact_freq[41:] = 0.05 * np.random.random(n_residues - 41)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# RMSD
axes[0,0].plot(t, rmsd, 'b-', alpha=0.7, linewidth=1)
axes[0,0].axvline(50, color='red', linestyle='--', label='Production start (50 ns)')
axes[0,0].axhline(2.5, color='orange', linestyle=':', label='2.5 Å threshold (pass)')
axes[0,0].set_xlabel('Time (ns)')
axes[0,0].set_ylabel('RMSD (Å)')
axes[0,0].set_title('Cα RMSD — Equilibration Check\n(plateau = equilibrated; drift = bad)')
axes[0,0].legend(fontsize=9)

# Helix fraction
axes[0,1].plot(t, helix_frac, 'g-', alpha=0.7, linewidth=1)
axes[0,1].axhline(0.6, color='red', linestyle='--', label='Threshold (60% helix)')
axes[0,1].set_xlabel('Time (ns)')
axes[0,1].set_ylabel('Fraction Helical')
axes[0,1].set_title('Secondary Structure Stability\n(DSSP helical fraction)')
axes[0,1].legend(fontsize=9)

# Lipid order parameter
axes[1,0].plot(carbon_pos, s_cd, 'ro-', label='GPCR simulation', linewidth=2)
axes[1,0].plot(carbon_pos, s_cd_ideal, 'b--', label='Experimental DPPC', linewidth=2)
axes[1,0].set_xlabel('Acyl Chain Carbon')
axes[1,0].set_ylabel('S_CD Order Parameter')
axes[1,0].set_title('Lipid Order Parameter\n(match experiment = good force field)')
axes[1,0].legend()

# Protein-lipid contacts
axes[1,1].bar(range(n_residues), contact_freq, color='purple', alpha=0.7)
axes[1,1].axvspan(14, 40, alpha=0.15, color='orange', label='Expected TM region')
axes[1,1].set_xlabel('Residue Index')
axes[1,1].set_ylabel('Lipid Contact Frequency')
axes[1,1].set_title('Protein-Lipid Contacts\n(TM region should dominate)')
axes[1,1].legend()

plt.tight_layout()
plt.savefig('/tmp/md_quality_checks.png', dpi=80, bbox_inches='tight')
plt.show()

# Summary
print("MD TRAJECTORY QUALITY REPORT")
print("=" * 40)
prod_rmsd = rmsd[50:]  # after equilibration
print(f"RMSD (production): mean={prod_rmsd.mean():.2f} Å, std={prod_rmsd.std():.2f} Å")
print(f"  {'✓ PASS' if prod_rmsd.std() < 0.5 else '✗ FAIL'}: RMSD variance {'acceptable' if prod_rmsd.std() < 0.5 else 'too large'}")
print()
print(f"Helix fraction (production): mean={helix_frac[50:].mean():.2f}")
print(f"  {'✓ PASS' if helix_frac[50:].mean() > 0.6 else '✗ FAIL'}: Secondary structure {'stable' if helix_frac[50:].mean() > 0.6 else 'unstable — bad for training data'}")
print()
tm_contacts = contact_freq[14:41].mean()
flank_contacts = np.concatenate([contact_freq[:14], contact_freq[41:]]).mean()
print(f"TM region lipid contacts: {tm_contacts:.2f} vs flanks: {flank_contacts:.2f}")
print(f"  {'✓ PASS' if tm_contacts > 3*flank_contacts else '✗ FAIL'}: Protein correctly embedded in bilayer")
print()
print("CONCLUSION: This trajectory is suitable for ML training (production frames > 50 ns)")


## Section 5 — ML Potentials: Replacing the Force Field

### Why Classical Force Fields Hit a Wall

The classical force field is **empirical** — its parameters were fit to small-molecule quantum chemistry calculations in the 1990s. It cannot describe:
- Bond breaking/formation (enzyme catalysis)
- Polarization effects (protein-ion binding)
- Non-equilibrium processes (ligand binding kinetics)

**ML potentials** train a neural network to predict the potential energy surface from atomic positions:

```
V = f_θ(atomic_positions, atomic_numbers)  ← neural network
F_i = -∂V/∂r_i                              ← forces by automatic differentiation
```

### Key ML Potentials (2024)

| Model | Architecture | Key Feature |
|-------|-------------|-------------|
| ANI-2x | Behler-Parrinello NN | First general organic chemistry ML potential |
| DeePMD | Deep Neural Network | DPMD + LiGeA, used for water/protein |
| NequIP | E(3)-equivariant GNN | Exact equivariance, sample-efficient |
| MACE | Higher-order equivariant | State-of-art accuracy, used in OpenMM |
| ESM-2 embeddings + head | Protein language model | Fine-tuned for ΔΔG (not MD, but related) |

**Connection to this curriculum:**
- NequIP/MACE use the same E(3)-equivariant architecture as SEGNN (Module 06)
- ESM-2 embeddings (Module 15) can be used as input features for ML potentials
- Uncertainty quantification (Module 13) applies directly to ML potential confidence


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ─────────────────────────────────────────────────────────────────────
# ML POTENTIAL CONCEPT: Symmetry-preserving energy prediction
# This is a simplified Behler-Parrinello-style descriptor
# Real: NequIP uses E(3)-equivariant convolutions (Module 06 connection)
# ─────────────────────────────────────────────────────────────────────

def radial_basis_function(r: np.ndarray, r_c: float = 6.0, n_basis: int = 8) -> np.ndarray:
    # Smooth cutoff function (goes to 0 at r_c)
    f_c = np.where(r < r_c, 0.5 * (np.cos(np.pi * r / r_c) + 1), 0.0)
    # Gaussian radial basis functions
    centers = np.linspace(0.5, r_c, n_basis)
    width = (r_c - 0.5) / n_basis
    rbf = np.exp(-((r[:, None] - centers[None, :]) / width)**2) * f_c[:, None]
    return rbf  # shape (n_neighbors, n_basis)

def symmetry_function(positions: np.ndarray, center_idx: int,
                      r_c: float = 6.0) -> np.ndarray:
    # Atom-centered symmetry function (Behler-Parrinello 2007)
    # Invariant to rotation, translation, permutation of same-element atoms
    center = positions[center_idx]
    neighbors = np.delete(positions, center_idx, axis=0)
    r_vecs = neighbors - center
    distances = np.linalg.norm(r_vecs, axis=1)

    # Only consider neighbors within cutoff
    mask = distances < r_c
    if not mask.any():
        return np.zeros(8)

    rbf = radial_basis_function(distances[mask], r_c=r_c)
    # Sum over neighbors: invariant to permutation
    descriptor = rbf.sum(axis=0)  # shape (n_basis,)
    return descriptor

# Test: symmetry functions should be invariant to rotation
n_atoms = 6
np.random.seed(42)
original_positions = np.random.randn(n_atoms, 3) * 3.0

# Random rotation matrix
theta = np.pi / 4
Rz = np.array([[np.cos(theta), -np.sin(theta), 0],
               [np.sin(theta),  np.cos(theta), 0],
               [0,              0,             1]])
rotated_positions = original_positions @ Rz.T

# Compute descriptors for atom 0
desc_original = symmetry_function(original_positions, center_idx=0)
desc_rotated = symmetry_function(rotated_positions, center_idx=0)
max_diff = np.abs(desc_original - desc_rotated).max()

print("ML POTENTIAL: SYMMETRY FUNCTION TEST")
print(f"  Descriptor before rotation: {desc_original.round(4)}")
print(f"  Descriptor after rotation:  {desc_rotated.round(4)}")
print(f"  Max difference: {max_diff:.2e}")
print(f"  {'✓ PASS' if max_diff < 1e-10 else '✗ FAIL'}: Descriptor is rotationally invariant")
print()

# Visualize: distance distribution → descriptor
distances_test = np.linspace(0.5, 6.0, 100)
rbf_test = radial_basis_function(distances_test, r_c=6.0, n_basis=8)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for i in range(8):
    axes[0].plot(distances_test, rbf_test[:, i], alpha=0.7, label=f'G_{i+1}')
axes[0].set_xlabel('Distance (Å)')
axes[0].set_ylabel('Basis function value')
axes[0].set_title('Radial Basis Functions (Behler-Parrinello)\nInvariant to rotation: only distances matter')
axes[0].legend(fontsize=7, ncol=2)

# Accuracy vs speed tradeoff
methods = ['AMBER ff19SB', 'CHARMM36m', 'ANI-2x', 'NequIP', 'MACE', 'DFT (B3LYP)']
accuracy = [2.0, 1.5, 0.8, 0.3, 0.2, 0.05]    # kcal/mol MAE
speed = [1000, 800, 50, 5, 3, 0.001]            # ns/day for 10k atom system

axes[1].scatter(speed, accuracy, s=100, c=['blue','blue','green','orange','red','purple'])
for i, m in enumerate(methods):
    axes[1].annotate(m, (speed[i], accuracy[i]), textcoords='offset points',
                     xytext=(5, 5), fontsize=9)
axes[1].set_xscale('log')
axes[1].set_xlabel('Simulation Speed (ns/day, 10k atoms)')
axes[1].set_ylabel('Energy MAE (kcal/mol)')
axes[1].set_title('Force Field Accuracy vs Speed\n(top-right = fast; bottom = accurate)')
axes[1].set_ylim(-0.1, 2.5)

plt.tight_layout()
plt.savefig('/tmp/md_ml_potential.png', dpi=80, bbox_inches='tight')
plt.show()

print("INSIGHT: NequIP and MACE achieve DFT-quality accuracy at 1000× DFT speed.")
print("Connection to Module 06: NequIP uses E(3)-equivariant message passing — exactly")
print("the same architectural principle as SEGNN for protein graphs.")


## Section 6 — Practical: Setting Up a Membrane Protein Simulation

### The GROMACS/OpenMM Workflow

For a real GPCR (e.g., β2-adrenergic receptor, PDB: 2RH1):

```bash
# Step 1: Download and prepare structure
wget https://files.rcsb.org/download/2RH1.pdb
python3 -m pdbfixer 2RH1.pdb --output=2rh1_fixed.pdb  # fix missing residues

# Step 2: Build membrane system with CHARMM-GUI (web tool)
# → Upload PDB to membrane.charmm-gui.org
# → Select "Membrane Builder" → POPC:POPE 3:1 bilayer
# → Download GROMACS topology files

# Step 3: Energy minimization (GROMACS)
gmx grompp -f minim.mdp -c system.gro -p topol.top -o em.tpr
gmx mdrun -v -deffnm em

# Step 4: Equilibration (2 stages)
gmx grompp -f nvt.mdp -c em.gro -r em.gro -p topol.top -o nvt.tpr
gmx mdrun -deffnm nvt
gmx grompp -f npt.mdp -c nvt.gro -r nvt.gro -p topol.top -o npt.tpr
gmx mdrun -deffnm npt

# Step 5: Production (100 ns+)
gmx grompp -f md.mdp -c npt.gro -p topol.top -o md.tpr
gmx mdrun -deffnm md -ntmpi 1 -ntomp 8 -gpu_id 0  # GPU-accelerated

# Step 6: Trajectory analysis with MDAnalysis
python3 analyze_gpcr.py
```

**OpenMM equivalent** (Python, easier for ML integration):
```python
from openmm.app import *
from openmm import *
from openmm.unit import *

pdb = PDBFile('2rh1_membrane.pdb')
forcefield = ForceField('charmm36.xml', 'charmm36/water.xml', 'charmm36/lipid.xml')
system = forcefield.createSystem(pdb.topology, nonbondedMethod=PME,
                                  nonbondedCutoff=1.2*nanometer,
                                  constraints=HBonds)
integrator = LangevinMiddleIntegrator(303.15*kelvin, 1/picosecond, 0.002*picoseconds)
simulation = Simulation(pdb.topology, system, integrator)
simulation.context.setPositions(pdb.positions)
simulation.reporters.append(DCDReporter('trajectory.dcd', 500))
simulation.step(50000000)  # 100 ns
```


> **Why this code:** This defines the gpcr_free_energy_landscape function, which we will call in later cells to perform a key step.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ─────────────────────────────────────────────────────────────────────
# SIMULATING GPCR ACTIVATION: Conformational state sampling
# Real activation: requires μs-scale MD or enhanced sampling
# Here: simplified energy landscape + Metropolis-Hastings sampling
# ─────────────────────────────────────────────────────────────────────

def gpcr_free_energy_landscape(tm3_displacement: np.ndarray,
                                 tm6_displacement: np.ndarray) -> np.ndarray:
    # 2D free energy landscape for GPCR activation
    # TM3 (inward/outward) × TM6 (inward/outward) displacement
    # Inactive state: TM3 in, TM6 in
    # Active state: TM3 out, TM6 out (large outward movement of TM6, ~14 Å)

    G = np.zeros_like(tm3_displacement)

    # Inactive state minimum (TM3 at -1, TM6 at -1)
    G += 2.0 * np.exp(-((tm3_displacement - (-1.5))**2 / (2 * 0.8**2) +
                         (tm6_displacement - (-1.0))**2 / (2 * 0.8**2)))

    # Active state minimum (TM3 at 0, TM6 at +2 — large TM6 movement)
    G += 3.0 * np.exp(-((tm3_displacement - 0.5)**2 / (2 * 0.6**2) +
                         (tm6_displacement - 2.0)**2 / (2 * 0.7**2)))

    # Intermediate state
    G += 1.0 * np.exp(-((tm3_displacement - (-0.5))**2 / (2 * 0.5**2) +
                         (tm6_displacement - 0.5)**2 / (2 * 0.5**2)))

    return -np.log(G + 1e-10)  # free energy = -kT * log(probability density)

x = np.linspace(-3, 2, 100)
y = np.linspace(-2, 3.5, 100)
X, Y = np.meshgrid(x, y)
G = gpcr_free_energy_landscape(X, Y)
G = G - G.min()  # normalize to minimum = 0

# Metropolis-Hastings sampling of the landscape
def sample_landscape(G_func, n_steps=10000, T_kT=1.0, step_size=0.3,
                     start=(-1.5, -1.0)):
    pos = np.array(list(start), dtype=float)
    traj = [pos.copy()]
    x_vals = np.linspace(-3, 2, 100)
    y_vals = np.linspace(-2, 3.5, 100)

    def get_G(p):
        xi = int((p[0] - (-3)) / 5 * 99)
        yi = int((p[1] - (-2)) / 5.5 * 99)
        xi = np.clip(xi, 0, 99)
        yi = np.clip(yi, 0, 99)
        return G[yi, xi]

    G_curr = get_G(pos)
    np.random.seed(42)
    for _ in range(n_steps):
        proposal = pos + np.random.normal(0, step_size, 2)
        G_prop = get_G(proposal)
        if np.log(np.random.random() + 1e-300) < -(G_prop - G_curr) / T_kT:
            pos = proposal
            G_curr = G_prop
        traj.append(pos.copy())
    return np.array(traj)

traj = sample_landscape(G, n_steps=5000, T_kT=0.8)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Free energy landscape + trajectory
contour = axes[0].contourf(X, Y, G, levels=20, cmap='RdYlBu_r', alpha=0.8)
plt.colorbar(contour, ax=axes[0], label='ΔG / kT')
axes[0].contour(X, Y, G, levels=10, colors='black', alpha=0.3, linewidths=0.5)
axes[0].scatter(traj[::5, 0], traj[::5, 1], c=range(len(traj[::5])),
                cmap='cool', s=5, alpha=0.5)
axes[0].scatter([-1.5], [-1.0], c='blue', s=200, marker='*', label='Inactive')
axes[0].scatter([0.5], [2.0], c='red', s=200, marker='*', label='Active')
axes[0].set_xlabel('TM3 displacement (Å)')
axes[0].set_ylabel('TM6 displacement (Å)')
axes[0].set_title('GPCR Activation Free Energy Landscape\n(β2AR-like; TM6 moves ~14 Å outward on activation)')
axes[0].legend()

# TM6 displacement over time
axes[1].plot(traj[:, 1], 'b-', alpha=0.6, linewidth=0.8)
axes[1].axhline(-1.0, color='blue', linestyle='--', alpha=0.5, label='Inactive TM6')
axes[1].axhline(2.0, color='red', linestyle='--', alpha=0.5, label='Active TM6')
axes[1].set_xlabel('MC Step')
axes[1].set_ylabel('TM6 Outward Displacement (Å)')
axes[1].set_title('TM6 Displacement vs Time\n(transitions between inactive and active)')
axes[1].legend()

plt.tight_layout()
plt.savefig('/tmp/md_gpcr_activation.png', dpi=80, bbox_inches='tight')
plt.show()

# Count transitions
n_inactive = np.sum(traj[:, 1] < 0.5)
n_active = np.sum(traj[:, 1] > 1.5)
print(f"GPCR STATE SAMPLING:")
print(f"  Inactive state (TM6 < 0.5 Å): {n_inactive} frames ({100*n_inactive/len(traj):.0f}%)")
print(f"  Active state (TM6 > 1.5 Å):   {n_active} frames ({100*n_active/len(traj):.0f}%)")
print()
print("LESSON: This is why you need μs-scale MD for GPCR activation — transitions are rare events.")
print("AF3 learns the STATIC active/inactive structures but NOT the transition pathway.")
print("ML + MD is needed to study kinetics (Module 12 connection: flow matching learns pathways).")


## 📄 Primary Literature & Resources — MD Simulation

### Foundational Papers
- **Shaw et al. 2010** — *Atomic-level characterization of the structural dynamics of proteins (Anton)*  
  [https://doi.org/10.1126/science.1187409](https://doi.org/10.1126/science.1187409)

- **Lindorff-Larsen et al. 2011** — *How fast-folding proteins fold (μs-scale MD)*  
  [https://doi.org/10.1126/science.1208351](https://doi.org/10.1126/science.1208351)

- **Batatia et al. 2022** — *MACE: Higher order equivariant message passing neural network potentials*  
  [https://arxiv.org/abs/2206.07697](https://arxiv.org/abs/2206.07697)

- **Batzner et al. 2022** — *NequIP: E(3)-equivariant graph neural networks for data-efficient ML force fields*  
  [https://doi.org/10.1038/s41467-022-29939-5](https://doi.org/10.1038/s41467-022-29939-5)

- **Huang et al. 2017** — *CHARMM36m: An improved force field for folded and intrinsically disordered proteins*  
  [https://doi.org/10.1038/nmeth.4067](https://doi.org/10.1038/nmeth.4067)

### Software & Databases
- **OpenMM documentation** — [https://openmm.org](https://openmm.org)  
  The leading Python-based MD engine; supports GPU acceleration and custom forces.

- **GPCRdb** — [https://gpcrdb.org](https://gpcrdb.org)  
  Curated database of GPCR structures, residue numbering, and pharmacology.

- **MemProtMD** — [https://memprotmd.bioch.ox.ac.uk](https://memprotmd.bioch.ox.ac.uk)  
  Oxford-hosted database of membrane protein MD simulations.

- **MDAnalysis** — [https://www.mdanalysis.org](https://www.mdanalysis.org)  
  Python library for analysing MD trajectories.

### Review Articles
- **Hollingsworth & Dror 2018** — *Molecular dynamics simulation for drug discovery*  
  [https://doi.org/10.1016/j.neuron.2018.08.011](https://doi.org/10.1016/j.neuron.2018.08.011)

- **Noé et al. 2020** — *Machine learning for molecular simulation (Annual Review)*  
  [https://doi.org/10.1146/annurev-physchem-042018-052331](https://doi.org/10.1146/annurev-physchem-042018-052331)


## Section 7 — Debug Exercises

### Exercise 1: Find the Bug in This MD Setup

```python
# BUGGY CODE — find and fix 3 bugs
def run_md_step(positions, velocities, forces, masses, dt=0.002):
    # Velocity Verlet
    accelerations = forces / masses
    new_positions = positions + velocities * dt  # Bug 1
    new_velocities = velocities + accelerations * dt  # Bug 2
    return new_positions, new_velocities

def compute_pbc_distance(r1, r2, box_length=60.0):
    dr = r2 - r1
    dr = dr % box_length  # Bug 3
    return np.linalg.norm(dr)
```

<details>
<summary>Show answers</summary>

**Bug 1:** Missing `0.5 * accelerations * dt**2` term. Correct: `positions + velocities * dt + 0.5 * accelerations * dt**2`

**Bug 2:** Velocity Verlet uses average acceleration from t and t+dt. Should compute new forces at new_positions first, then: `velocities + 0.5 * (accelerations + new_accelerations) * dt`

**Bug 3:** `%` doesn't give minimum image. Should be: `dr - box_length * np.round(dr / box_length)` to put dr in [-L/2, +L/2]
</details>

### Exercise 2: Trajectory Quality Check

Your colleague gives you an MD trajectory with these properties:
- RMSD (production frames): mean = 2.1 Å, std = 1.8 Å
- Helix fraction: drops from 0.68 to 0.31 over 200 ns
- S_CD (carbon 8): 0.05 (literature: 0.20 ± 0.02)
- TM contact frequency: uniform across all residues (0.15 ± 0.03)

**Question:** Which force field was likely used and what's wrong?

<details>
<summary>Show answer</summary>

1. **RMSD std = 1.8 Å** — too high, protein is unfolding (normal: < 0.5 Å)
2. **Helix fraction drop** — protein is unfolding during production (never add these frames to ML training data)
3. **S_CD = 0.05** — bilayer is disordered (experimental DPPC: 0.20). Likely used AMBER ff19SB without lipid parameters
4. **Uniform TM contacts** — protein escaped the bilayer (no longer embedded)

**Root cause:** AMBER ff19SB used for a membrane protein. Lipid parameters are absent → bilayer collapses → protein unfolds. Should use CHARMM36m.
</details>


## 📚 Resources

### Primary References
- **CHARMM-GUI** (`membrane.charmm-gui.org`) — the standard tool for setting up membrane simulations, no coding required
- **MDAnalysis tutorial** (`userguide.mdanalysis.org`) — Python MD trajectory analysis; Jupyter-friendly
- **OpenMM documentation** (`openmm.org/documentation`) — Python MD engine with GPU acceleration
- **GROMACS tutorial (Lemkul)** (`mdtutorials.com`) — the canonical free tutorial; covers lysozyme + DPPC membrane

### University Courses
- **MIT 8.592J** — Statistical Physics in Biology (OCW free) — covers statistical mechanics behind MD sampling
- **MIT 7.91J** — Computational Biology (OCW free) — includes protein simulation unit
- **UCSD Biophysics** — GPCR structural biology (Bhaskara Rao, YouTube lectures)

### Papers to Read
- Noe et al. 2020, *Science*: "Boltzmann generators: Sampling equilibrium states of many-body systems" — ML for enhanced sampling
- Jumper et al. 2021, *Nature*: AlphaFold2 supplementary Section 1.9 — how they used MD for training data curation
- Scheen et al. 2020 — CHARMM36m validation for membrane protein simulations

### ML Potentials
- **NequIP paper** (Batzner et al. 2022, *Nature Comm*) — equivariant neural network interatomic potential
- **MACE** (Batatia et al. 2023) — current state-of-art; available on GitHub
- **OpenMM-ML** — plug-in to use ML potentials inside OpenMM simulations

### Cross-Module Connections
- **Module 03** (RMSD, Kabsch): understanding MD trajectory alignment before RMSD calculation
- **Module 06** (GNNs, equivariance): NequIP and MACE use E(3)-equivariant GNNs — same math
- **Module 12** (Diffusion): flow matching models learn paths between MD conformations
- **Module 13** (Bayesian): uncertainty in ML potential predictions → MC Dropout on energy heads


## Mastery Check

After completing this notebook, you should be able to:

1. **Write** a Verlet integrator from scratch (tested in Exercise 1)
2. **Explain** why CHARMM36m is required for membrane protein MD (not AMBER)
3. **Identify** 4 quality checks for an MD trajectory before using it for ML training
4. **Describe** how NequIP's architecture connects to the E(3)-equivariant GNNs in Module 06
5. **Distinguish** what MD gives you that AF3 doesn't (conformational dynamics, kinetics, free energy)

### Interview Q&A

**Q:** A collaborator ran a 500 ns simulation of a GPCR with AMBER ff14SB and says the protein is "unstable." What's the first thing you check?
> **A:** Force field choice. AMBER ff14SB has no lipid parameters — the bilayer will disorder and the protein will unfold. Re-run with CHARMM36m. Also check: was PME used for long-range electrostatics? Was the system equilibrated with position restraints first?

**Q:** Why can't you just train AF3 on MD trajectory frames instead of experimental PDB structures?
> **A:** Several reasons: (1) MD force fields have systematic errors (~2 kcal/mol MAE vs experiment); (2) MD frames are temporally correlated — naive splitting leaks information; (3) coarse-grained MD has unphysical features; (4) the most interesting conformations (transition states, rare events) are statistically underrepresented. MD data is most useful for augmentation, not sole training source.

**Q:** How does flow matching (Module 12) connect to MD?
> **A:** Flow matching learns a vector field interpolating between two distributions — it can learn to map from a Gaussian prior to the equilibrium distribution of protein conformations (which MD samples). Recent work (AlphaFlow, EigenFold) conditions diffusion on sequence to generate conformational ensembles without running MD.


## Notebook Complete

**You can now:**
- Describe the Verlet integrator, periodic boundary conditions, and PME electrostatics
- Compare classical force fields with ML potentials for molecular dynamics

**Where this fits:**
- Previous: 01_membrane_protein_dynamics
- Next: 12_generative_models/01_diffusion_protein_design — Module 12 — check the README

**Before moving on, check:**
- [ ] All code cells ran without errors
- [ ] You understand what each function does (read the docstrings)
- [ ] You can explain the key concept in one sentence without looking at notes

## 🎤 Interview Questions

**Q1 (Easy):** Why can't we use an arbitrarily large timestep in molecular dynamics?
<details><summary>Answer</summary>
The timestep must be smaller than the fastest motion in the system. For all-atom MD, the fastest vibrations are bond stretches (especially C-H bonds) with periods around 10 femtoseconds. A timestep of 1-2 fs samples these motions adequately. Larger timesteps cause energy conservation violations and the simulation "explodes" (atoms overlap, forces diverge). Constraint algorithms (SHAKE, LINCS) freeze fast bond vibrations, allowing 2 fs timesteps; hydrogen mass repartitioning (HMR) enables 4 fs.
</details>

**Q2 (Easy):** What are periodic boundary conditions (PBC) and why are they necessary?
<details><summary>Answer</summary>
PBC replicate the simulation box infinitely in all directions, so atoms leaving one side re-enter from the opposite side. This eliminates surface artifacts — without PBC, molecules at the box edge would experience vacuum, which is unphysical for bulk simulations. PBC approximate an infinite system using a finite box. The minimum image convention ensures each atom interacts only with the closest copy of every other atom.
</details>

**Q3 (Medium):** Explain the difference between the NVT and NPT ensembles and when you would use each.
<details><summary>Answer</summary>
NVT (canonical): constant Number of particles, Volume, and Temperature — controlled by a thermostat. NPT (isothermal-isobaric): constant N, Pressure, and Temperature — controlled by thermostat + barostat, allowing volume to fluctuate. Use NVT for gas-phase or fixed-volume calculations. Use NPT for condensed-phase simulations (proteins in water) because it allows the box to adjust to the correct density. Typical protocol: energy minimisation → NVT equilibration (heat up) → NPT equilibration (adjust density) → NPT production.
</details>

**Q4 (Medium):** How does PME (Particle Mesh Ewald) handle long-range electrostatics, and why is it essential?
<details><summary>Answer</summary>
Electrostatic interactions decay as 1/r — very slowly, so truncation at a cutoff introduces large errors. PME splits the calculation: short-range part computed directly in real space (within cutoff), long-range part computed in reciprocal (Fourier) space using FFTs. The charges are interpolated onto a grid, FFT-transformed, multiplied by the Ewald kernel, and inverse-FFT'd. Complexity: O(N log N) instead of O(N²) for direct summation. Without PME, membrane simulations and any system with significant charges would produce artifacts.
</details>

**Q5 (Hard):** You want to study a conformational change in a protein that occurs on the millisecond timescale, but your MD budget is microseconds. What enhanced sampling methods could you use?
<details><summary>Answer</summary>
Methods: (1) Replica Exchange MD (REMD/T-REMD): run multiple replicas at different temperatures, swap configurations; high-T replicas cross barriers, low-T replicas sample. (2) Metadynamics: add bias potential along collective variables (CVs) to fill free energy wells and escape barriers; recover the free energy surface. (3) Accelerated MD (aMD/GaMD): boost the potential energy to reduce barriers while maintaining a reweighting scheme. (4) Adaptive sampling: run many short simulations, use Markov State Models (MSMs) to identify under-sampled states, and launch new simulations from those states. (5) String method / transition path sampling for specific pathways. Key challenge: choosing good collective variables for metadynamics. MSMs + adaptive sampling are CV-free and increasingly popular. For ML integration: train neural network potentials on ab initio data for faster force evaluation, or use ML-guided adaptive sampling.
</details>